# Problema secuencial: ¿hacia dónde se mueve la pelota?
## ConvLSTM vs. CNN vs. ANN

Cada ejemplo es un **video corto** de 10 cuadros de 20×20 píxeles. En el video hay:

* una **pelota que se mueve** en línea recta hacia **arriba, abajo, izquierda o derecha**,
* una **pelota quieta** (distractor) que no aporta información,
* **ruido** en cada cuadro.

**Tarea:** clasificar la dirección del movimiento (4 clases).

¿Por qué es **secuencial**? Con **un solo cuadro** es imposible saber la dirección: la respuesta está en
**cómo cambia la imagen en el tiempo**. Al final lo comprobamos desordenando los cuadros.

| Modelo | Cómo ve el video | ¿Espacio? | ¿Tiempo? |
|---|---|---|---|
| **ANN** (red densa) | aplana los 10×20×20 = 4000 píxeles en un vector | ✗ | ✗ (solo posiciones del vector) |
| **CNN** (2D) | apila los 10 cuadros como 10 canales de una imagen | ✓ | parcial (los mezcla todos en la 1.ª capa) |
| **ConvLSTM** | lee los cuadros **uno por uno** y guarda una **memoria espacial** | ✓ | ✓ |

> En Colab: `Entorno de ejecución → Ejecutar todo`. Con GPU (T4) tarda ~1-2 min; en CPU unos minutos.

## 1. Librerías y parámetros

In [ ]:
# @title Importar librerías { display-mode: "form" }
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

keras.utils.set_random_seed(42)
CLASES = ["arriba", "abajo", "izquierda", "derecha"]
DIRECCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # (dy, dx) de cada clase

In [ ]:
# @title Parámetros { display-mode: "form" }
CUADROS = 10  # @param {type:"integer"}
TAM = 20  # @param {type:"integer"}
N_ENTRENAMIENTO = 3000  # @param {type:"integer"}
N_PRUEBA = 600  # @param {type:"integer"}
RUIDO = 0.25  # @param {type:"slider", min:0.0, max:0.6, step:0.05}
EPOCAS = 15  # @param {type:"slider", min:5, max:40, step:1}

## 2. Generar los videos
Cada cuadro es una imagen en escala de grises. La pelota es una mancha gaussiana; en cada cuadro avanza
`velocidad` píxeles en su dirección (velocidad aleatoria entre 0.8 y 1.5). La forma de los datos es
`(videos, cuadros, alto, ancho, canales)` = `(N, 10, 20, 20, 1)`.

In [ ]:
# @title Generador de videos { display-mode: "form" }
def generar_videos(n, cuadros=CUADROS, tam=TAM, ruido=RUIDO, semilla=0):
    rng = np.random.default_rng(semilla)
    yy, xx = np.mgrid[0:tam, 0:tam]
    mancha = lambda py, px: np.exp(-((yy - py) ** 2 + (xx - px) ** 2) / (2 * 1.2 ** 2))
    X = np.zeros((n, cuadros, tam, tam, 1), dtype="float32")
    y = rng.integers(0, 4, n)
    lo, hi = 2, tam - 3  # márgenes para que la pelota no salga
    for i in range(n):
        dy, dx = DIRECCIONES[y[i]]
        vel = rng.uniform(0.8, 1.5)
        recorrido = vel * (cuadros - 1)
        def inicio(d):  # punto de partida para que toda la trayectoria quepa
            if d > 0:
                return rng.uniform(lo, hi - recorrido)
            if d < 0:
                return rng.uniform(lo + recorrido, hi)
            return rng.uniform(lo, hi)
        cy, cx = inicio(dy), inicio(dx)
        ey, ex = rng.uniform(lo, hi, 2)  # pelota quieta (distractor)
        for t in range(cuadros):
            cuadro = mancha(cy + dy * vel * t, cx + dx * vel * t) + 0.7 * mancha(ey, ex)
            cuadro += ruido * rng.standard_normal((tam, tam))
            X[i, t, :, :, 0] = np.clip(cuadro, 0, 1)
    return X, y


X_ent, y_ent = generar_videos(N_ENTRENAMIENTO, semilla=1)
X_pru, y_pru = generar_videos(N_PRUEBA, semilla=2)
print("Entrenamiento:", X_ent.shape, " Prueba:", X_pru.shape)
print("Videos por clase (entrenamiento):", dict(zip(CLASES, np.bincount(y_ent))))

In [ ]:
# @title Ver un video de cada clase { display-mode: "form" }
fig, ejes = plt.subplots(4, CUADROS + 1, figsize=(1.4 * (CUADROS + 1), 6.2))
for c in range(4):
    i = np.where(y_ent == c)[0][0]
    for t in range(CUADROS):
        ejes[c, t].imshow(X_ent[i, t, :, :, 0], cmap="gray", vmin=0, vmax=1)
        ejes[c, t].set_title(f"t={t}", fontsize=8)
    # "Estela": cada cuadro pintado con un color según el tiempo
    estela = np.max(X_ent[i, :, :, :, 0] * np.linspace(0.2, 1, CUADROS)[:, None, None], axis=0)
    ejes[c, -1].imshow(estela, cmap="viridis")
    ejes[c, -1].set_title("estela", fontsize=8)
    ejes[c, 0].set_ylabel(CLASES[c], fontsize=11)
for a in ejes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("Cada fila es un video. La dirección solo se ve comparando cuadros en el tiempo", y=1.0)
plt.tight_layout(); plt.show()

## 3. Los tres modelos
Los tres terminan igual (capa densa + **softmax de 4 clases**) para que la comparación sea justa.

**ConvLSTM:** es una LSTM donde las multiplicaciones de matrices se cambian por **convoluciones**.
En cada cuadro $t$ recibe la imagen $X_t$ y su estado anterior $H_{t-1}$ (mapas de características), y con
**compuertas** (olvido, entrada, salida) decide qué recordar en su **memoria de celda** $C_t$, que también es una imagen:

$$i_t=\sigma(W_{xi}*X_t+W_{hi}*H_{t-1}+b_i)\quad f_t=\sigma(W_{xf}*X_t+W_{hf}*H_{t-1}+b_f)$$
$$C_t=f_t\odot C_{t-1}+i_t\odot\tanh(W_{xc}*X_t+W_{hc}*H_{t-1}+b_c)\quad o_t=\sigma(\dots)\quad H_t=o_t\odot\tanh(C_t)$$

($*$ = convolución, $\odot$ = producto elemento a elemento). Así recuerda **dónde estaba** la pelota y lo compara con
**dónde está ahora**.

In [ ]:
# @title Construir ANN, CNN y ConvLSTM { display-mode: "form" }
FORMA = (CUADROS, TAM, TAM, 1)


def crear_ann():
    return keras.Sequential([
        keras.Input(FORMA),
        layers.Flatten(),                       # 10*20*20 = 4000 entradas
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(4, activation="softmax"),
    ], name="ANN")


def crear_cnn():
    return keras.Sequential([
        keras.Input(FORMA),
        layers.Permute((2, 3, 1, 4)),           # (alto, ancho, cuadros, 1)
        layers.Reshape((TAM, TAM, CUADROS)),    # los 10 cuadros como 10 canales
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(4, activation="softmax"),
    ], name="CNN")


def crear_convlstm():
    return keras.Sequential([
        keras.Input(FORMA),
        layers.ConvLSTM2D(16, 3, padding="same", return_sequences=True),   # lee cuadro por cuadro
        layers.BatchNormalization(),
        layers.ConvLSTM2D(16, 3, padding="same", return_sequences=False),  # solo el estado final H_T
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(4, activation="softmax"),
    ], name="ConvLSTM")


modelos = {"ANN": crear_ann(), "CNN": crear_cnn(), "ConvLSTM": crear_convlstm()}
for m in modelos.values():
    m.summary()

## 4. Entrenar los tres modelos con los mismos datos y épocas

In [ ]:
# @title Entrenar { display-mode: "form" }
historias, tiempos = {}, {}
for nombre, m in modelos.items():
    print(f"\n===== {nombre} =====")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    historias[nombre] = m.fit(X_ent, y_ent, validation_split=0.2, epochs=EPOCAS,
                              batch_size=64, verbose=2).history
    tiempos[nombre] = time.time() - t0

## 5. Comparar resultados

In [ ]:
# @title Tabla comparativa { display-mode: "form" }
predicciones, filas = {}, []
for nombre, m in modelos.items():
    predicciones[nombre] = m.predict(X_pru, verbose=0).argmax(1)
    filas.append({
        "Modelo": nombre,
        "Parámetros": m.count_params(),
        "Exactitud entrenamiento": historias[nombre]["accuracy"][-1],
        "Exactitud prueba": (predicciones[nombre] == y_pru).mean(),
        "Tiempo (s)": tiempos[nombre],
    })
tabla = pd.DataFrame(filas).set_index("Modelo")
display(tabla.style.format({"Parámetros": "{:,}", "Exactitud entrenamiento": "{:.1%}",
                            "Exactitud prueba": "{:.1%}", "Tiempo (s)": "{:.1f}"}))

In [ ]:
# @title Curvas de entrenamiento { display-mode: "form" }
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for nombre, h in historias.items():
    linea, = a1.plot(h["val_accuracy"], label=f"{nombre} (validación)")
    a1.plot(h["accuracy"], "--", color=linea.get_color(), alpha=0.5)
    a2.plot(h["val_loss"], label=nombre, color=linea.get_color())
a1.axhline(0.25, color="gray", ls=":", label="azar (25%)")
a1.set(title="Exactitud (línea punteada = entrenamiento)", xlabel="época", ylim=(0, 1.02)); a1.legend()
a2.set(title="Pérdida en validación", xlabel="época"); a2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# @title Matrices de confusión { display-mode: "form" }
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.3))
for a, (nombre, p) in zip(ejes, predicciones.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_pru, p), display_labels=CLASES).plot(
        ax=a, cmap="Blues", colorbar=False)
    a.set_title(f"{nombre}: {(p == y_pru).mean():.1%}")
plt.tight_layout(); plt.show()

In [ ]:
# @title Ejemplos de prueba con la predicción de cada modelo { display-mode: "form" }
N_EJEMPLOS = 8  # @param {type:"integer"}
fig, ejes = plt.subplots(1, N_EJEMPLOS, figsize=(2.2 * N_EJEMPLOS, 3.4))
for k, a in enumerate(np.atleast_1d(ejes)):
    estela = np.max(X_pru[k, :, :, :, 0] * np.linspace(0.2, 1, CUADROS)[:, None, None], axis=0)
    a.imshow(estela, cmap="viridis"); a.set_xticks([]); a.set_yticks([])
    texto = [f"Real: {CLASES[y_pru[k]]}"]
    for nombre, p in predicciones.items():
        texto.append(f"{nombre}: {CLASES[p[k]]} {'✓' if p[k] == y_pru[k] else '✗'}")
    a.set_title("\n".join(texto), fontsize=8)
plt.suptitle("Estela del video (oscuro = inicio, claro = final)")
plt.tight_layout(); plt.show()

## 6. ¿De verdad es un problema secuencial?
Si **barajamos el orden de los cuadros**, las imágenes son las mismas pero el orden se pierde.
Si los modelos dependen del tiempo, su exactitud debe caer al **azar (≈25%)**.

In [ ]:
# @title Probar con los cuadros desordenados { display-mode: "form" }
rng = np.random.default_rng(0)
X_barajado = np.stack([v[rng.permutation(CUADROS)] for v in X_pru])
filas = []
for nombre, m in modelos.items():
    p = m.predict(X_barajado, verbose=0).argmax(1)
    filas.append({"Modelo": nombre, "Orden correcto": (predicciones[nombre] == y_pru).mean(),
                  "Cuadros barajados": (p == y_pru).mean()})
display(pd.DataFrame(filas).set_index("Modelo").style.format("{:.1%}"))

## 7. Conclusiones para la clase
* **ANN:** aplana el video y pierde la estructura espacial y temporal; tiene que memorizar qué combinación de
  píxeles corresponde a cada dirección. Muchos parámetros (la primera capa es 4000×128) y es la que más se sobreajusta,
  sobre todo con más ruido.
* **CNN 2D:** sí aprovecha la vecindad de los píxeles y, al poner los cuadros como canales, puede comparar cuadros,
  pero mezcla todo el tiempo en la primera convolución y queda atada a un número fijo de cuadros.
* **ConvLSTM:** procesa los cuadros **en orden** con una memoria que conserva la forma de la imagen; es la que mejor
  modela "dónde estaba → dónde está", con muy pocos parámetros. A cambio, es más lenta de entrenar (no se paraleliza en el tiempo).
* Al barajar los cuadros los tres modelos caen cerca del azar: la información está en la **secuencia**.

**Para experimentar:** sube el `RUIDO` (0.4–0.5) o baja `N_ENTRENAMIENTO` (500) y vuelve a ejecutar: la diferencia
entre ConvLSTM y la ANN se hace más grande.